# Public Trade Analysis

Loads raw trades from `data/trades_raw/`, grouped by **wallet × market × timestamp**,
and enriches each trade with the **final token value** (winner / loser / price at resolution).

In [ ]:
import json
import datetime
from pathlib import Path

import pandas as pd

## Parameters

In [ ]:
# ── date range (inclusive) of market folders to load ──────────────────────────
START_DATE     = datetime.date(2026, 1, 1)
END_DATE       = datetime.date(2026, 3, 6)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 1)

# ── path to trades_raw ────────────────────────────────────────────────────────
TRADES_RAW = Path("../data/trades_raw")

# ── output parquet ────────────────────────────────────────────────────────────
TRADES_PARQUET = Path("../data/trades_processed.parquet")

## 1 · Load markets and trades

In [ ]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

In [ ]:
def load_market(json_path: Path) -> dict:
    """Load a single market definition from its .json file."""
    with json_path.open("rb") as f:
        return orjson.loads(f.read()) if "orjson" in dir() else json.load(f)


def load_trades(jsonl_path: Path) -> list[dict]:
    """Load all trades from a .jsonl file into a list of dicts."""
    rows = []
    with jsonl_path.open("rb") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json_loads(line)
            obj["asset"] = str(obj["asset"])
            rows.append(obj)
    return rows


def day_folders(root: Path, start: datetime.date, end: datetime.date) -> list[Path]:
    """Return sorted list of YYYY-MM-DD folders within [start, end]."""
    folders = []
    for p in sorted(root.iterdir()):
        if not p.is_dir():
            continue
        try:
            d = datetime.date.fromisoformat(p.name)
        except ValueError:
            continue
        if start <= d <= end:
            folders.append(p)
    return folders

In [ ]:
folders = day_folders(TRADES_RAW, START_DATE, END_DATE)
print(f"Found {len(folders)} day-folder(s): {[f.name for f in folders]}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

def load_market_and_trades(json_path: Path) -> tuple[dict, list[dict]] | None:
    """Load market definition and its trades from a single json/jsonl pair."""
    jsonl_path = json_path.with_suffix(".jsonl")
    if not jsonl_path.exists():
        return None
    
    market = load_market(json_path)
    trades = load_trades(jsonl_path)
    return market, trades


# collect all json file paths
all_json_paths = [
    json_path
    for folder in folders
    for json_path in sorted(folder.glob("*.json"))
]

all_trades: list[dict] = []
markets: dict[str, dict] = {}

num_workers = min(32, os.cpu_count() * 4 or 16)  # more threads for I/O-bound work

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(load_market_and_trades, p): p for p in all_json_paths}
    
    for future in as_completed(futures):
        result = future.result()
        if result is None:
            continue
        
        market, trades = result
        condition_id = market["condition_id"]
        
        # keep only the first time we see a condition_id
        if condition_id not in markets:
            markets[condition_id] = market
        
        all_trades.extend(trades)

print(f"Loaded {len(all_trades):,} trade records across {len(markets):,} unique markets.")

## 2 · Build the trades DataFrame

In [ ]:
df = pd.DataFrame(all_trades)

# parse timestamp → UTC datetime (vectorized)
df["dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# canonical column names
df.rename(columns={"proxyWallet": "wallet", "conditionId": "condition_id"}, inplace=True)

# ensure numeric types (use pd.to_numeric for better performance)
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# sort chronologically (consider if you really need this - it's expensive)
df.sort_values(["condition_id", "wallet", "dt"], inplace=True, ignore_index=True)

print(df.shape)
df.head(3)

## 3 · Enrich with market metadata and final token value

For each trade we add:

| column | description |
|---|---|
| `question` | Human-readable market question |
| `end_date` | Market resolution date |
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Final settlement price of the traded token (1.0 = winner, 0.0 = loser, or last known price if unresolved) |
| `trade_value_usdc` | `size × price` — USDC spent on the trade |
| `final_value_usdc` | `size × final_price` — USDC value of shares at resolution |

In [ ]:
def build_token_lookup(markets: dict[str, dict]) -> dict[str, dict]:
    """Return {token_id → {condition_id, outcome, winner, final_price}} for all markets."""
    lookup: dict[str, dict] = {}
    for cid, m in markets.items():
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # if market is closed/resolved use binary settlement, else use last price
            if m.get("closed", False):
                final_price = 1.0 if winner else 0.0
            else:
                final_price = float(tok.get("price") or 0.0)
            lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }
    return lookup


token_lookup = build_token_lookup(markets)
print(f"Token lookup entries: {len(token_lookup):,}")

In [ ]:
# join token resolution data
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "asset"
token_df.reset_index(inplace=True)

df = df.merge(
    token_df[["asset", "token_winner", "final_price"]],
    on="asset",
    how="left",
)

# join market-level metadata
market_meta = pd.DataFrame(
    [
        {
            "condition_id": cid,
            "question":     m["question"],
            "end_date":     pd.to_datetime(m["end_date_iso"], utc=True),
            "market_slug":  m["market_slug"],
        }
        for cid, m in markets.items()
    ]
)

df = df.merge(market_meta, on="condition_id", how="left")

# derived value columns
df["trade_value_usdc"] = df["size"] * df["price"]
df["final_value_usdc"] = df["size"] * df["final_price"]

print(df.shape)
df[["wallet", "condition_id", "dt", "side", "outcome", "size", "price",
    "token_winner", "final_price", "trade_value_usdc", "final_value_usdc"]].head(5)

## 4 · Group by wallet × market × timestamp

Each row in `grouped` represents all trades made by one wallet in one market at
one exact timestamp (i.e. a single on-chain transaction can place multiple fills).

In [ ]:
GROUP_KEYS = ["wallet", "condition_id", "dt"]

grouped = (
    df.groupby(GROUP_KEYS, sort=False)
    .agg(
        question          = ("question",          "first"),
        market_slug       = ("market_slug",        "first"),
        end_date          = ("end_date",           "first"),
        side              = ("side",               "first"),   # BUY / SELL
        outcome           = ("outcome",            "first"),   # Yes / No
        token_winner      = ("token_winner",       "first"),
        final_price       = ("final_price",        "first"),
        # sum across fills in the same tx
        total_size        = ("size",               "sum"),
        avg_price         = ("price",              "mean"),
        trade_value_usdc  = ("trade_value_usdc",   "sum"),
        final_value_usdc  = ("final_value_usdc",   "sum"),
        num_fills         = ("transactionHash",    "count"),
    )
    .reset_index()
    .sort_values(["wallet", "condition_id", "dt"])
    .reset_index(drop=True)
)

print(f"Grouped rows: {len(grouped):,}  (from {len(df):,} raw trades)")
grouped.head(10)

## 5 · Wallet-level P&L summary (training data only)

Aggregate per wallet using **training trades only** (`dt.date <= END_DATE_TRAIN`).
This ensures the top-5% selection is not contaminated by future (test) data.

In [ ]:
# sign convention: BUY spends USDC, SELL receives USDC
grouped["signed_cost"] = grouped.apply(
    lambda r: r["trade_value_usdc"] if r["side"] == "BUY" else -r["trade_value_usdc"],
    axis=1,
)
grouped["signed_final"] = grouped.apply(
    lambda r: r["final_value_usdc"] if r["side"] == "BUY" else -r["final_value_usdc"],
    axis=1,
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_date_train_ts = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < end_date_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets       = ("condition_id",  "nunique"),
        num_trades        = ("num_fills",      "sum"),
        total_cost_usdc   = ("signed_cost",    "sum"),
        total_final_usdc  = ("signed_final",   "sum"),
    )
    .assign(pnl_usdc=lambda x: x["total_final_usdc"] - x["total_cost_usdc"] )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(2)

## 6 · Market-level volume summary

In [ ]:
market_summary = (
    grouped.groupby(["condition_id", "question", "end_date", "token_winner"])
    .agg(
        num_wallets      = ("wallet",            "nunique"),
        num_trades       = ("num_fills",          "sum"),
        volume_usdc      = ("trade_value_usdc",   "sum"),
        final_value_usdc = ("final_value_usdc",   "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [ ]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

## 8 · Full enriched trade table

The main output: one row per wallet × market × timestamp, with all enrichment columns.

In [ ]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt", "end_date",
    "question", "side", "outcome", "total_size", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "num_fills",
]

grouped[DISPLAY_COLS]

## 9 · Export top-5% wallets to parquet

Identifies top-5% wallets by P&L on the **training period**, then exports their
trades from the **full dataset** (training + test) to `data/trades_processed.parquet`.

Each row is a single trade fill enriched with `trade_value_usdc`, `final_value_usdc`,
and a boolean `is_train` flag that marks whether the trade falls in the training window.

In [ ]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
for i in range(0, 11):
    q = i / 10
    pnl_q = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"].quantile(q)
    print(f"  P&L at {q:.0%} pct: {pnl_q:,.2f} USDC")

In [ ]:
EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "asset", "outcome", "outcomeIndex",
    "size", "price",
    "trade_value_usdc", "final_value_usdc",
    "token_winner", "final_price",
    "name", "pseudonym", "transactionHash", "title",
    "is_train",
]

# ── filter full dataset (all dates) to top wallets ────────────────────────────
top_trades = df[df["wallet"].isin(top_wallets)].copy()

# ── flag training vs test rows ────────────────────────────────────────────────
top_trades["is_train"] = top_trades["dt"].dt.date <= END_DATE_TRAIN

print(f"Total trade rows:  {len(top_trades):,}")
print(f"  training rows:   {top_trades['is_train'].sum():,}")
print(f"  test rows:       {(~top_trades['is_train']).sum():,}")

# ── write to parquet ──────────────────────────────────────────────────────────
TRADES_PARQUET.parent.mkdir(parents=True, exist_ok=True)
top_trades[EXPORT_COLS].to_parquet(TRADES_PARQUET, index=False)
print(f"\nSaved {len(top_trades):,} rows → {TRADES_PARQUET}")